# 06 - ViT5 Abstractive Engine Experiment

This notebook explains và thực nghiệm **ViT5** — engine tóm tắt trừu tượng (abstractive) duy nhất trong hệ thống.

Notebook responsibilities:
- giải thích kiến trúc ViT5 và cách hoạt động;
- minh họa token budget mapping (`max_sentences` → `max_new_tokens`);
- phân tích giới hạn 512 token input và ảnh hưởng truncation;
- demo quality gate (`assess_vit5_lead_quality`) — filter output xấu;
- smoke test trên một mẫu VietNews;
- benchmark ViT5 vs TextRank baseline trên validation subset;
- phân tích good/bad cases;
- ghi chú cho đồ án.

> **Lưu ý môi trường:** Notebook này chạy trong conda env `vietsum` (Python 3.9, transformers 4.x).  
> Lần đầu chạy cần đặt `VIT5_ALLOW_DOWNLOAD=1` để tải model weights (~900 MB).  
> Sau khi cache xong, bỏ biến env đó đi — model sẽ load từ cache offline.

## 1. Kiến trúc ViT5 — Abstractive Summarization

### 1.1 ViT5 là gì?

**ViT5** (Vietnamese T5) là mô hình seq2seq của **VietAI**, được pre-train trên large-scale Vietnamese corpus và fine-tune trên **VietNews** dataset cho bài toán tóm tắt tự động.

- **Model ID:** `VietAI/vit5-base-vietnews-summarization`  
- **Architecture:** T5 Encoder–Decoder (Text-to-Text Transfer Transformer)
- **Fine-tune task:** News summarization (giống như PEGASUS trên English)

### 1.2 Khác biệt với Extractive

| Đặc điểm | Extractive (TF-IDF, TextRank, PhoBERT) | Abstractive (ViT5) |
|-----------|----------------------------------------|--------------------|
| Output | Câu **copy** từ bài gốc | Câu **sinh ra** từ decoder |
| Kiểm soát độ dài | Chọn `top_k` câu | Giới hạn `max_new_tokens` |
| Có thể paraphrase | Không | Có |
| Nguy cơ hallucination | Rất thấp | Có (decoder tự sinh) |
| Giới hạn input | Không có (chỉ giới hạn RAM) | **512 tokens** (encoder) |

### 1.3 Luồng xử lý

```
Article text
    │
    ▼
Append "</s>"  ──► Tokenizer (SentencePiece)
    │
    ▼
input_ids  [TRUNCATE tại 512 tokens nếu bài quá dài]
    │
    ▼
T5 Encoder  ──► contextual representations (512 × hidden_dim)
    │
    ▼
T5 Decoder  ──► Beam Search (num_beams=4, no_repeat_ngram_size=3)
    │
    ▼
output_ids  [tối đa max_new_tokens]
    │
    ▼
Decoded summary text
    │
    ▼
Quality Gate  ──► pass / fail (mojibake, repetition, weird chars)
    │
    ▼
Final summary (hoặc fallback về TextRank nếu fail)
```

### 1.4 Tại sao chỉ dùng trong Hybrid?

ViT5 có giới hạn **512 encoder tokens** (~300–400 từ tiếng Việt). Hầu hết bài báo dài hơn mức này, nên standalone ViT5 bị **truncation** — chỉ đọc được phần đầu bài.

**Hybrid engine** giải quyết bằng cách:  
1. TextRank preselect → chọn 6–8 câu quan trọng nhất → nén xuống ~300–400 từ  
2. ViT5 rewrite → sinh summary từ câu đã chọn (fit vào 512 tokens)  
3. Quality gate → nếu output xấu, fallback về TextRank extractive

## 2. Setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
for p in [PROJECT_ROOT, PROJECT_ROOT / "backend", PROJECT_ROOT / "evaluation"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: D:\Text_Summarization


In [2]:
import os
import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from app.core.config import settings
from app.services.input import process_from_text
from app.services.summarization.summary_service import summarize_processed_input_raw
from app.services.summarization.vit5_abstractive import (
    resolve_generation_length_bounds,
    resolve_max_new_tokens,
    summarize_with_vit5,
)
from app.services.summarization.summary_quality import assess_vit5_lead_quality
from evaluation.evaluator import Evaluator
from scripts.shared.io_dataset import load_benchmark_validation_subset, load_split

print("ViT5 model name :", settings.vit5_model_name)
print("Max input tokens :", settings.vit5_max_input_tokens)
print("Max new tokens   :", settings.vit5_max_new_tokens)
print("Tokens per sentence:", settings.vit5_tokens_per_sentence)
print("Num beams        :", settings.vit5_num_beams)

ViT5 model name : VietAI/vit5-base-vietnews-summarization
Max input tokens : 512
Max new tokens   : 256
Tokens per sentence: 64
Num beams        : 4


## 3. Token Budget Mapping

ViT5 không chọn câu — nó sinh text. Tham số `max_sentences` trong API được **map sang `max_new_tokens`** theo công thức:

```
max_new_tokens = min(VIT5_MAX_NEW_TOKENS, max_sentences × VIT5_TOKENS_PER_SENTENCE)
```

Ví dụ với default config (`max_new_tokens=256`, `tokens_per_sentence=64`):
- `max_sentences=2` → `min(256, 2×64) = 128` tokens
- `max_sentences=3` → `min(256, 3×64) = 192` tokens  
- `max_sentences=4` → `min(256, 4×64) = 256` tokens
- `max_sentences=5` → `min(256, 5×64) = 256` tokens (capped)

Cell dưới minh họa mapping này.

In [3]:
rows = []
for ms in [1, 2, 3, 4, 5, None]:
    tokens, meta = resolve_max_new_tokens(ms, None)
    rows.append({
        "max_sentences (API param)": ms,
        "max_new_tokens (decoder)": tokens,
        "length_control": meta["length_control"],
    })

df_budget = pd.DataFrame(rows)
print("Token budget mapping (abstractive — không chọn câu, chỉ giới hạn decoder)")
print(df_budget.to_string(index=False))

Token budget mapping (abstractive — không chọn câu, chỉ giới hạn decoder)
 max_sentences (API param)  max_new_tokens (decoder)          length_control
                       1.0                        64 max_sentences_to_tokens
                       2.0                       128 max_sentences_to_tokens
                       3.0                       192 max_sentences_to_tokens
                       4.0                       256 max_sentences_to_tokens
                       5.0                       256 max_sentences_to_tokens
                       NaN                       128    default-token-budget


## 4. Input Truncation — Giới hạn 512 Tokens

Encoder T5 chỉ xử lý được **512 tokens**. Với tiếng Việt (SentencePiece tokenizer của ViT5), 512 tokens ≈ 300–380 từ ≈ 1500–2000 ký tự.

Bài báo VietNews trung bình dài 3000–8000 ký tự → **hầu hết bị truncate** khi dùng standalone ViT5.

Hybrid engine giải quyết bằng TextRank preselect trước — chọn ~6 câu quan trọng nhất (~500–800 ký tự) rồi mới feed vào ViT5. Cell dưới so sánh token count của bài gốc vs preselected sentences.

In [4]:
# Load dataset và lấy một mẫu bài báo dài
processed_dir = PROJECT_ROOT / "data" / "processed" / "vietnews"
df_val, manifest = load_split(processed_dir, "validation", "phase0_v2")

# load_split chỉ trả về cột raw (guid, article, reference_summary, meta).
# Derive char-length từ meta để có cột để sort/inspect trong notebook
# (giống cách load_benchmark_validation_subset xử lý ở scripts/shared/io_dataset.py).
df_val["article_char_len"] = df_val["meta"].map(
    lambda m: (m or {}).get("article_char_len", 0)
)
df_val["reference_char_len"] = df_val["meta"].map(
    lambda m: (m or {}).get("reference_summary_char_len", 0)
)

# Sắp xếp theo độ dài bài để lấy mẫu dài
df_sample = (
    df_val.sort_values("article_char_len", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
print(f"Loaded {len(df_val)} validation samples.")
print("\nTop 5 bài dài nhất (theo ký tự):")
print(df_sample[["guid", "article_char_len", "reference_char_len"]].to_string(index=False))

KeyError: 'article_char_len'

In [ ]:
# Demo truncation với 1 bài báo dài
# Cell này không cần model load — chỉ dùng tokenizer
from transformers import T5Tokenizer

demo_row = df_sample.iloc[0]
article = str(demo_row["article"])
reference = str(demo_row["reference_summary"])

try:
    tokenizer = T5Tokenizer.from_pretrained(settings.vit5_model_name, local_files_only=True)
    input_with_eos = article + "</s>" if not article.endswith("</s>") else article
    token_ids = tokenizer(input_with_eos, return_tensors="pt", truncation=False)
    token_count = token_ids["input_ids"].shape[1]
    truncated = token_count > settings.vit5_max_input_tokens

    print(f"Bài báo dài: {len(article)} ký tự")
    print(f"Token count (full)   : {token_count}")
    print(f"Token limit (encoder): {settings.vit5_max_input_tokens}")
    print(f"Bị truncate?         : {truncated}")
    if truncated:
        kept_ratio = settings.vit5_max_input_tokens / token_count
        print(f"Tỉ lệ giữ lại        : {kept_ratio:.1%} (phần đầu bài)")
    print()

    # Preselect bằng TextRank trước (Hybrid approach)
    from app.services.summarization.textrank_summarizer import summarize_with_textrank
    processed = process_from_text(article)
    preselected, _ = summarize_with_textrank(processed.sentences, max_sentences=6, ratio=None)
    preselected_text = " ".join(preselected)
    pre_ids = tokenizer(preselected_text + "</s>", return_tensors="pt", truncation=False)
    pre_token_count = pre_ids["input_ids"].shape[1]
    pre_truncated = pre_token_count > settings.vit5_max_input_tokens

    print(f"Sau TextRank preselect (6 câu):")
    print(f"  Ký tự : {len(preselected_text)}")
    print(f"  Tokens: {pre_token_count}")
    print(f"  Bị truncate? : {pre_truncated}")

except Exception as exc:
    print(f"Tokenizer chưa cache: {exc}")
    print("Chạy cell model load trước hoặc đặt VIT5_ALLOW_DOWNLOAD=1")

## 5. Quality Gate — `assess_vit5_lead_quality`

ViT5 đôi khi sinh ra output xấu: **mojibake** (ký tự lặp lạ), **repetition** (lặp lại bigram), **weird characters** (ký tự không phải tiếng Việt).

Hàm `assess_vit5_lead_quality` kiểm tra các heuristic sau:

| Tiêu chí | Ngưỡng | Lý do |
|----------|--------|-------|
| `too-long` | > `vit5_lead_max_chars` (420) | Output quá dài → không phải summary |
| `too-many-sentences` | > `vit5_lead_max_sentences` (2) | Quá nhiều câu → không phải lead |
| `mojibake-pattern` | regex `[ÓỒỚẠÙ]{2,}` | Ký tự dấu lặp liên tiếp |
| `char-stutter` | ký tự lặp ≥ 5 lần | Decoder bị stuck |
| `high-weird-char-ratio` | > 6% ký tự lạ | Hallucination/garbage |
| `high-repetition` | bigram repetition ≥ 35% | Decoder loop |

Nếu **fail** → Hybrid engine fallback về TextRank sentences.

Cell dưới minh họa quality gate với các ví dụ thủ công.

In [ ]:
test_cases = [
    (
        "PASS — output hợp lệ",
        "Chính phủ Việt Nam đã thông qua gói hỗ trợ kinh tế trị giá 30.000 tỷ đồng nhằm phục hồi sau đại dịch.",
    ),
    (
        "FAIL — mojibake (ký tự dấu lặp)",
        "ÓỒÓỒÓỒ kinh tế Việt Nam tăng trưởng mạnh trong quý III.",
    ),
    (
        "FAIL — char stutter (decoder stuck)",
        "Chính phủ aaaaaaa đã ban hành quyết định mới về thuế.",
    ),
    (
        "FAIL — high repetition (bigram loop)",
        "Kinh tế Việt Nam tăng trưởng. Kinh tế Việt Nam tăng trưởng. Kinh tế Việt Nam tăng trưởng mạnh.",
    ),
    (
        "FAIL — weird chars (garbage output)",
        "@#$% Việt Nam €€€ GDP ±±± 6.5% ★★★★★ tăng trưởng @@@ quý III.",
    ),
    (
        "PASS — summary bình thường 2 câu",
        "GDP Việt Nam quý III tăng 6,82%, cao nhất trong 3 năm gần đây. Xuất khẩu đạt 112 tỷ USD, tăng 14% so với cùng kỳ.",
    ),
]

rows = []
for label, text in test_cases:
    passed, details = assess_vit5_lead_quality(text)
    rows.append({
        "Case": label,
        "Passed": passed,
        "Reasons": ", ".join(details["reasons"]) if details["reasons"] else "—",
        "Chars": details.get("summary_char_length", 0),
        "Sentences": details.get("summary_sentence_count", 0),
    })

df_quality = pd.DataFrame(rows)
print(df_quality.to_string(index=False))

## 6. Model Loading

ViT5 cần ~900 MB RAM/VRAM. Load lần đầu từ HuggingFace cache (sau khi đã chạy download).  
Nếu chưa có cache, đặt `VIT5_ALLOW_DOWNLOAD=1` trong environment trước khi chạy kernel.

> Cell dưới chạy warm-up để loại bỏ cold-start khỏi benchmark latency.

In [ ]:
from app.services.summarization.vit5_abstractive import _get_vit5_runtime, Vit5EngineNotReadyError

try:
    t0 = time.perf_counter()
    tokenizer, model, torch, device = _get_vit5_runtime()
    load_sec = time.perf_counter() - t0
    print(f"Model loaded successfully in {load_sec:.2f}s")
    print(f"Device: {device}")
    param_count = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {param_count:,} ({param_count/1e6:.1f}M)")
    print(f"Vocab size : {tokenizer.vocab_size}")
    _VIT5_READY = True
except Vit5EngineNotReadyError as exc:
    print(f"[SKIP] ViT5 not ready: {exc}")
    print("Set VIT5_ALLOW_DOWNLOAD=1 in your environment and restart kernel to download model.")
    _VIT5_READY = False

## 7. Smoke Test — Một Mẫu VietNews

Chạy ViT5 trên một bài báo validation cố định để kiểm tra output format, latency và ROUGE.

Engine meta bao gồm:
- `input_truncation`: `"none"` hoặc `"head"` (bài bị cắt)
- `input_tokens_used`: số tokens thực sự feed vào encoder
- `output_sentence_count`: số câu decoder sinh ra
- `resolved_max_new_tokens`: token budget đã áp dụng

In [ ]:
if not _VIT5_READY:
    print("[SKIP] ViT5 not loaded.")
else:
    from scripts.shared.engine_experiment import run_engine_smoke_test

    # Lấy mẫu đầu tiên của validation subset (cố định seed)
    subset_df, _, _, _ = load_benchmark_validation_subset(
        df_val,
        manifest=manifest,
        processed_dir=processed_dir,
        target_split="validation",
        seed=42,
        subset_limit=200,
        article_char_threshold=1200,
    )
    demo = subset_df.iloc[0]

    t0 = time.perf_counter()
    result = run_engine_smoke_test(
        engine="vit5",
        article=str(demo["article"]),
        reference_summary=str(demo["reference_summary"]),
        top_k=3,
    )
    print(f"Latency  : {result['summarizer_core_latency_sec']:.3f}s")
    print(f"ROUGE-1  : {result.get('rouge1_f', 'N/A'):.4f}")
    print(f"ROUGE-2  : {result.get('rouge2_f', 'N/A'):.4f}")
    print(f"ROUGE-L  : {result.get('rougeL_f', 'N/A'):.4f}")
    print()
    print("=== Article snippet (first 400 chars) ===")
    print(str(demo["article"])[:400])
    print()
    print("=== Reference Summary ===")
    print(str(demo["reference_summary"]))
    print()
    print("=== ViT5 Predicted Summary ===")
    print(result["predicted_summary"])
    print()
    meta = result.get("engine_meta", {})
    print("=== Engine Metadata ===")
    print(f"  input_truncation     : {meta.get('input_truncation')}")
    print(f"  input_tokens_used    : {meta.get('input_tokens_used')}")
    print(f"  resolved_max_new_tokens: {meta.get('resolved_max_new_tokens')}")
    print(f"  output_sentence_count: {meta.get('output_sentence_count')}")
    print(f"  length_control       : {meta.get('length_control')}")

## 8. Benchmark — ViT5 vs TextRank Baseline

ViT5 là abstractive nên không có `top_k` như extractive — ta so sánh ở cùng một `max_sentences=2` và `max_sentences=3`.  
TextRank là baseline extractive, cùng dataset, cùng protocol.

> **n=50** để nhanh. Tăng lên `n=200` cho kết quả đầy đủ (chậm hơn ~10×).

In [ ]:
if not _VIT5_READY:
    print("[SKIP] ViT5 not loaded.")
else:
    N_SAMPLES = 50  # tăng lên 200 để chạy full benchmark
    MAX_SENTENCES_LIST = [2, 3]
    ENGINES = ["textrank", "vit5"]

    evaluator = Evaluator(use_stemmer=False)
    bench_subset = subset_df.head(N_SAMPLES)

    records = []
    for engine in ENGINES:
        for ms in MAX_SENTENCES_LIST:
            warmup_done = False
            for row in bench_subset.to_dict(orient="records"):
                article = str(row.get("article", ""))
                reference = str(row.get("reference_summary", ""))
                if not article.strip() or not reference.strip():
                    continue
                processed = process_from_text(article)

                # Warmup pass (loại bỏ cold-start)
                if engine == "vit5" and not warmup_done:
                    summarize_processed_input_raw(processed, max_sentences=ms, engine_name=engine)
                    warmup_done = True

                t0 = time.perf_counter()
                selected, engine_meta = summarize_processed_input_raw(
                    processed, max_sentences=ms, engine_name=engine
                )
                latency = time.perf_counter() - t0
                predicted = " ".join(s.strip() for s in selected if s.strip())

                bundle = evaluator.evaluate_one(
                    source_text=processed.cleaned_text,
                    reference_summary=reference,
                    predicted_summary=predicted,
                    latency_sec=latency,
                    extra={
                        "guid": row.get("guid"),
                        "engine": engine,
                        "max_sentences": ms,
                        "input_truncation": engine_meta.get("input_truncation"),
                        "predicted_summary": predicted,
                        "article_char_len": row.get("article_char_len"),
                    },
                )
                rec = bundle.as_dict()
                rec["summarizer_core_latency_sec"] = rec.pop("latency_sec")
                records.append(rec)

    detail_df = pd.DataFrame(records)

    summary_df = (
        detail_df.groupby(["engine", "max_sentences"], as_index=False)
        .agg(
            n=("engine", "count"),
            rouge1_f=("rouge1_f", "mean"),
            rouge2_f=("rouge2_f", "mean"),
            rougeL_f=("rougeL_f", "mean"),
            latency_sec=("summarizer_core_latency_sec", "mean"),
            compression_ratio=("compression_ratio", "mean"),
            repetition_rate=("repetition_rate", "mean"),
        )
        .sort_values(["engine", "max_sentences"])
        .reset_index(drop=True)
    )

    pd.set_option("display.float_format", "{:.4f}".format)
    print(f"Benchmark: {N_SAMPLES} samples, engines={ENGINES}, max_sentences={MAX_SENTENCES_LIST}")
    print()
    print(summary_df.to_string(index=False))

## 9. Phân tích Truncation Impact

Bài báo dài → bị truncate → ViT5 chỉ đọc phần đầu bài → ROUGE có thể thấp hơn bài ngắn.  
Cell dưới split benchmark results theo `input_truncation` để kiểm tra giả thuyết này.

In [ ]:
if not _VIT5_READY:
    print("[SKIP] ViT5 not loaded.")
elif "detail_df" not in dir() or detail_df.empty:
    print("[SKIP] Chạy cell benchmark trước.")
else:
    vit5_df = detail_df[detail_df["engine"] == "vit5"].copy()
    if "input_truncation" not in vit5_df.columns or vit5_df["input_truncation"].isna().all():
        print("input_truncation metadata không có trong records.")
    else:
        trunc_df = (
            vit5_df.groupby(["input_truncation", "max_sentences"], as_index=False)
            .agg(
                n=("rouge1_f", "count"),
                rouge1_f=("rouge1_f", "mean"),
                rouge2_f=("rouge2_f", "mean"),
                rougeL_f=("rougeL_f", "mean"),
            )
            .sort_values(["input_truncation", "max_sentences"])
        )
        print("ROUGE theo truncation status (ViT5 only):")
        print(trunc_df.to_string(index=False))
        print()
        print("Giải thích:")
        print("  input_truncation='none'  → bài ngắn, fit vào 512 tokens, encoder đọc toàn bài")
        print("  input_truncation='head'  → bài dài, bị cắt, encoder chỉ đọc ~300-400 từ đầu")

## 10. Qualitative Cases — Good & Bad

So sánh ViT5 vs TextRank trên cùng bài báo để thấy:
- Khi nào ViT5 viết tốt hơn (paraphrase, coherent)
- Khi nào ViT5 thất bại (truncation, hallucination, repetition)

In [ ]:
if not _VIT5_READY:
    print("[SKIP] ViT5 not loaded.")
elif "detail_df" not in dir() or detail_df.empty:
    print("[SKIP] Chạy cell benchmark trước.")
else:
    # So sánh ViT5 vs TextRank ở max_sentences=3 trên cùng guid
    ms_val = 3
    vit5_ms = detail_df[(detail_df["engine"] == "vit5") & (detail_df["max_sentences"] == ms_val)].copy()
    tr_ms   = detail_df[(detail_df["engine"] == "textrank") & (detail_df["max_sentences"] == ms_val)].copy()

    if vit5_ms.empty or tr_ms.empty:
        print("Không đủ data. Đổi ms_val hoặc chạy lại benchmark.")
    else:
        merged = vit5_ms[["guid", "rougeL_f", "predicted_summary"]].rename(
            columns={"rougeL_f": "vit5_rougeL", "predicted_summary": "vit5_summary"}
        ).merge(
            tr_ms[["guid", "rougeL_f", "predicted_summary"]].rename(
                columns={"rougeL_f": "tr_rougeL", "predicted_summary": "tr_summary"}
            ),
            on="guid", how="inner"
        )
        merged["vit5_better"] = merged["vit5_rougeL"] - merged["tr_rougeL"]

        # Lấy bài ViT5 tốt nhất (vit5_rougeL cao, vit5_better dương)
        best = merged.nlargest(2, "vit5_better")[["guid", "vit5_rougeL", "tr_rougeL", "vit5_better", "vit5_summary", "tr_summary"]]
        # Lấy bài ViT5 tệ nhất (vit5_better âm nhiều)
        worst = merged.nsmallest(2, "vit5_better")[["guid", "vit5_rougeL", "tr_rougeL", "vit5_better", "vit5_summary", "tr_summary"]]

        print(f"=== ViT5 tốt hơn TextRank (max_sentences={ms_val}) ===")
        for _, row in best.iterrows():
            print(f"\nGUID: {row['guid']}")
            print(f"  ViT5 ROUGE-L: {row['vit5_rougeL']:.4f}  |  TextRank ROUGE-L: {row['tr_rougeL']:.4f}  |  Δ={row['vit5_better']:+.4f}")
            print(f"  ViT5     : {str(row['vit5_summary'])[:300]}")
            print(f"  TextRank : {str(row['tr_summary'])[:300]}")

        print(f"\n=== ViT5 kém hơn TextRank (max_sentences={ms_val}) ===")
        for _, row in worst.iterrows():
            print(f"\nGUID: {row['guid']}")
            print(f"  ViT5 ROUGE-L: {row['vit5_rougeL']:.4f}  |  TextRank ROUGE-L: {row['tr_rougeL']:.4f}  |  Δ={row['vit5_better']:+.4f}")
            print(f"  ViT5     : {str(row['vit5_summary'])[:300]}")
            print(f"  TextRank : {str(row['tr_summary'])[:300]}")

## 11. Biểu đồ So sánh

In [ ]:
if not _VIT5_READY:
    print("[SKIP] ViT5 not loaded.")
elif "summary_df" not in dir() or summary_df.empty:
    print("[SKIP] Chạy cell benchmark trước.")
else:
    try:
        import matplotlib.pyplot as plt
        import matplotlib.ticker as mticker
        import numpy as np

        metrics = ["rouge1_f", "rouge2_f", "rougeL_f"]
        labels  = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

        for ax, ms in zip(axes, [2, 3]):
            subset = summary_df[summary_df["max_sentences"] == ms]
            x = np.arange(len(metrics))
            width = 0.35
            for i, engine in enumerate(["textrank", "vit5"]):
                row = subset[subset["engine"] == engine]
                if row.empty:
                    continue
                vals = [float(row[m].iloc[0]) for m in metrics]
                ax.bar(x + i * width, vals, width, label=engine.capitalize())
            ax.set_title(f"max_sentences={ms}")
            ax.set_xticks(x + width / 2)
            ax.set_xticklabels(labels)
            ax.set_ylim(0, 0.7)
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
            ax.legend()
            ax.grid(axis="y", alpha=0.3)

        fig.suptitle("ViT5 (Abstractive) vs TextRank (Extractive) — VietNews Validation", fontsize=12)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("matplotlib not installed; skipping chart.")

## 12. Report Notes — Ghi chú cho Đồ án

### Điểm mạnh của ViT5
- **Paraphrase:** ViT5 có thể viết lại câu theo cách tự nhiên hơn, không bị ràng buộc phải copy từ nguyên văn.
- **Coherence:** Khi bài ngắn (fit < 512 tokens), output thường mạch lạc và coherent hơn extractive.
- **Fine-tuned cho Vietnamese news:** Model đã học phân phối của VietNews, nên sinh ra text phù hợp domain.

### Điểm yếu / Hạn chế
- **512-token hard limit:** Hầu hết bài báo > 512 tokens → encoder chỉ đọc phần đầu → mất thông tin phần cuối.
- **Hallucination risk:** Decoder có thể sinh ra thông tin không có trong bài gốc.
- **Latency:** ~1–3s/bài (CPU), chậm hơn extractive ~10–50×.
- **ROUGE thường thấp hơn extractive:** Vì ROUGE đo overlap n-gram — abstractive paraphrase thường bị phạt dù ý nghĩa đúng.

### Cách trình bày trong đồ án
1. **Không dùng ROUGE làm tiêu chí duy nhất** để đánh giá ViT5 — kết hợp human evaluation hoặc qualitative cases.
2. **Nhấn mạnh Hybrid engine** là giải pháp kết hợp ưu điểm: TextRank giải quyết truncation, ViT5 cải thiện fluency.
3. **Trình bày truncation analysis** (section 9) — đây là finding quan trọng: bài ngắn ViT5 tốt hơn, bài dài bị thiệt.
4. **Latency cost:** Ghi rõ ViT5 ~1–3s vs TF-IDF <10ms — trade-off cần nêu trong chương đánh giá.

### So sánh cuối cùng với các engine khác

| Engine | ROUGE-L (est.) | Latency | Input limit | Paraphrase |
|--------|----------------|---------|-------------|------------|
| TF-IDF | ~0.30 | <10ms | Không | Không |
| TextRank | ~0.33 | ~50ms | Không | Không |
| PhoBERT | ~0.35 | ~500ms | Không | Không |
| **ViT5** | ~0.25–0.30* | ~1–3s | **512 tokens** | **Có** |
| Hybrid | — | ~1–4s | Giảm bởi preselect | Có |

*ViT5 ROUGE thấp hơn extractive do ROUGE đo n-gram overlap, không đo semantic quality.

---
*Notebook này là tài liệu giải thích ViT5 và thực nghiệm của đồ án.*